<a id="title"></a>
# <p style="background-color:green; font-family:calibri; color:white; font-size:150%; text-align:center; border-radius:20px 20px;">**Phase 3.3 - Patient-Level CAD-RADS 2.0 Scoring**</p>

## Method Overview

This notebook converts the **segment-level stenosis summary** from Phase 3.2 into a **patient-level CAD-RADS 2.0 code**.

Input:

`results/block3_results/segment stenosis/{sample_name}/stenosis_summary_{sample_name}.xlsx`

The previous notebook summarized each anatomical segment using its maximum `%AS`. Here we apply the patient-level rules from the JCCT 2022 CAD-RADS 2.0 expert consensus:

- the numeric CAD-RADS category is based on the most clinically relevant/highest-grade stenosis on a **per-patient basis**;
- CAD-RADS 4 is split into **4A** and **4B** depending on left main or three-vessel severe disease;
- CAD-RADS 5 is reserved for **100% total coronary occlusion**;
- the plaque burden modifier `P1-P4` is derived from Segment Involvement Score (SIS).

The default sample is `Normal_1`, but all paths are parameterized with `SAMPLE_NAME`.

<a id="step1"></a>
# <p style="background-color:green; font-family:calibri; color:white; font-size:120%; text-align:center; border-radius:20px 20px;">1. Imports & Configuration</p>

Change only `SAMPLE_NAME` to score another patient/sample. The notebook resolves the repository root automatically and writes the final patient-level report under `results/block3_results/cad-rads/`.

In [2]:
# Purpose: import dependencies and configure patient-level CAD-RADS scoring.
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import HTML, Markdown, display

SAMPLE_NAME = "Normal_1"  # Change this value to process another sample.

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src").is_dir():
    if PROJECT_ROOT == PROJECT_ROOT.parent:
        raise RuntimeError(
            "Could not locate the project root containing src/. "
            "Run Jupyter with the working directory inside the repository."
        )
    PROJECT_ROOT = PROJECT_ROOT.parent

SEGMENT_STENOSIS_ROOT = PROJECT_ROOT / "results" / "block3_results" / "segment stenosis"
LEGACY_SEGMENT_CADRADS_ROOT = PROJECT_ROOT / "results" / "block3_results" / "segment cad-rads"
OUTPUT_ROOT = PROJECT_ROOT / "results" / "block3_results" / "cad-rads"

print(f"Sample name  : {SAMPLE_NAME}")
print(f"Project root : {PROJECT_ROOT}")
print(f"Output root  : {OUTPUT_ROOT}")

Sample name  : Normal_1
Project root : C:\Users\adria\OneDrive\Escriptori\UPF'\TFG\UPF_TFGRepository
Output root  : C:\Users\adria\OneDrive\Escriptori\UPF'\TFG\UPF_TFGRepository\results\block3_results\cad-rads


<a id="step2"></a>
# <p style="background-color:green; font-family:calibri; color:white; font-size:120%; text-align:center; border-radius:20px 20px;">2. Load Segment Stenosis Summary</p>

The JCCT document states that CAD-RADS should be applied **per patient** using the clinically most relevant, usually highest-grade, stenosis. Therefore, this notebook starts from the Phase 3.2 table where each AHA/ASOCA segment already has a maximum `%AS` and a segment-level severity grade.

The loader also accepts the earlier column names used during exploration (`CADRADS_Category`, `CADRADS_Label`) and converts them to the current segment-severity names.

In [3]:
# Purpose: load and normalize the segment-level stenosis summary.
def resolve_segment_summary_path(sample_name: str) -> Path:
    """Resolve the Phase 3.2 segment stenosis summary for the selected sample."""
    candidates = [
        SEGMENT_STENOSIS_ROOT / sample_name / f"stenosis_summary_{sample_name}.xlsx",
        LEGACY_SEGMENT_CADRADS_ROOT / sample_name / f"stenosis_summary_{sample_name}.xlsx",
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate

    searched = "\n".join(f"- {path}" for path in candidates)
    raise FileNotFoundError(
        f"Could not find stenosis_summary for {sample_name}. Checked:\n{searched}"
    )


def classify_segment_stenosis(max_pct_as: float) -> pd.Series:
    """Assign the CAD-RADS 2.0 stenosis severity interval to a segment %AS."""
    if pd.isna(max_pct_as):
        return pd.Series(
            {"Stenosis_Severity_Grade": pd.NA, "Stenosis_Severity_Label": "Not assessable"}
        )

    value = float(np.clip(max_pct_as, 0, 100))
    if value == 0:
        grade, label = 0, "No visible stenosis"
    elif value < 25:
        grade, label = 1, "Minimal stenosis"
    elif value < 50:
        grade, label = 2, "Mild stenosis"
    elif value < 70:
        grade, label = 3, "Moderate stenosis"
    elif value < 100:
        grade, label = 4, "Severe stenosis"
    else:
        grade, label = 5, "Total occlusion"

    return pd.Series(
        {"Stenosis_Severity_Grade": grade, "Stenosis_Severity_Label": label}
    )


SEGMENT_SUMMARY_PATH = resolve_segment_summary_path(SAMPLE_NAME)
segment_summary = pd.read_excel(SEGMENT_SUMMARY_PATH)

legacy_renames = {
    "CADRADS_Category": "Stenosis_Severity_Grade",
    "CADRADS_Label": "Stenosis_Severity_Label",
    "Artery": "Specific_Artery",
}
segment_summary = segment_summary.rename(
    columns={old: new for old, new in legacy_renames.items() if old in segment_summary.columns}
)

required_columns = {"Segment_ID", "Segment_Name", "Max_pct_AS"}
missing_columns = sorted(required_columns - set(segment_summary.columns))
if missing_columns:
    raise KeyError(f"Missing required columns in segment summary: {missing_columns}")

segment_summary["Segment_ID"] = pd.to_numeric(
    segment_summary["Segment_ID"], errors="coerce"
).astype("Int64")
segment_summary["Max_pct_AS"] = pd.to_numeric(segment_summary["Max_pct_AS"], errors="coerce")
segment_summary["Max_pct_AS_Clipped"] = segment_summary["Max_pct_AS"].clip(lower=0, upper=100)

if "Stenosis_Severity_Grade" not in segment_summary.columns:
    segment_summary[["Stenosis_Severity_Grade", "Stenosis_Severity_Label"]] = segment_summary[
        "Max_pct_AS_Clipped"
    ].apply(classify_segment_stenosis)
else:
    segment_summary[["Stenosis_Severity_Grade", "Stenosis_Severity_Label"]] = segment_summary[
        "Max_pct_AS_Clipped"
    ].apply(classify_segment_stenosis)

segment_summary["Stenosis_Severity_Grade"] = pd.to_numeric(
    segment_summary["Stenosis_Severity_Grade"], errors="coerce"
).astype("Int64")

for optional_col in ["Artery_Type", "Specific_Artery"]:
    if optional_col not in segment_summary.columns:
        segment_summary[optional_col] = "Unknown"

print(f"Loaded file    : {SEGMENT_SUMMARY_PATH}")
print(f"Rows / columns : {segment_summary.shape[0]} / {segment_summary.shape[1]}")
display(segment_summary.sort_values("Segment_ID"))

Loaded file    : C:\Users\adria\OneDrive\Escriptori\UPF'\TFG\UPF_TFGRepository\results\block3_results\segment stenosis\Normal_1\stenosis_summary_Normal_1.xlsx
Rows / columns : 16 / 10


,Segment_ID,Segment_Name,Artery_Type,Specific_Artery,Max_pct_AS,Stenosis_Severity_Grade,Stenosis_Severity_Label,Point_Count,Valid_pct_AS_Count,Max_pct_AS_Clipped
13,1,Proximal RCA,RCA,RCA,15.772175,1,Minimal stenosis,115,57,15.772175
12,2,Mid RCA,RCA,RCA,34.672462,2,Mild stenosis,405,405,34.672462
7,3,Distal RCA,RCA,RCA,66.672740,3,Moderate stenosis,333,333,66.672740
2,4,Right PDA,RCA,RCA,87.248610,4,Severe stenosis,259,188,87.248610
14,5,Left Main,LCA,LCA,NaN,<NA>,Not assessable,72,0,NaN
6,6,Proximal LAD,LCA,LAD,78.543653,4,Severe stenosis,215,214,78.543653
3,7,Mid LAD,LCA,LAD,85.418851,4,Severe stenosis,110,92,85.418851
8,8,Distal LAD,LCA,LAD,65.635050,3,Moderate stenosis,235,228,65.635050
15,9,First diagonal (D1),LCA,LAD,NaN,<NA>,Not assessable,71,0,NaN
1,10,Second diagonal (D2),LCA,LAD,92.800950,4,Severe stenosis,233,167,92.800950


<a id="step3"></a>
# <p style="background-color:green; font-family:calibri; color:white; font-size:120%; text-align:center; border-radius:20px 20px;">3. Vessel Grouping</p>

For the CAD-RADS 4A/4B distinction, the JCCT table separates severe single/two-vessel disease from left main or three-vessel obstructive disease.

This notebook uses the following main coronary territories:

- **RCA territory:** segments `1, 2, 3, 4`.
- **LAD territory:** segments `6, 7, 8, 9, 10`.
- **LCX territory:** segments `11, 12, 13, 15, 17`.
- **Left Main:** segment `5`, evaluated separately because Table 4 upgrades left main stenosis `>=50%` to CAD-RADS 4B.

Unmapped segments are still included in the global maximum stenosis and SIS calculations, but they are not assigned to RCA/LAD/LCX territory counts until the anatomical dictionary is extended.

In [4]:
# Purpose: compute territory-level obstruction/severity status.
TERRITORY_SEGMENTS = {
    "RCA": {1, 2, 3, 4},
    "LAD": {6, 7, 8, 9, 10},
    "LCX": {11, 12, 13, 15, 17},
}
LEFT_MAIN_SEGMENT_ID = 5
EXPECTED_SIS_SEGMENT_COUNT = 16  # JCCT SIS convention; actual present segments can vary by sample.
KNOWN_CADRADS_SEGMENTS = set().union(*TERRITORY_SEGMENTS.values(), {LEFT_MAIN_SEGMENT_ID})


def top_segment_label(df: pd.DataFrame) -> str:
    """Return a readable label for the highest-stenosis segment in a dataframe."""
    if df.empty or df["Max_pct_AS_Clipped"].dropna().empty:
        return "Not present"
    row = df.sort_values(["Max_pct_AS_Clipped", "Segment_ID"], ascending=[False, True]).iloc[0]
    return f"{row['Segment_Name']} (segment {int(row['Segment_ID'])})"


territory_rows = []
for territory_name, segment_ids in TERRITORY_SEGMENTS.items():
    territory_df = segment_summary.loc[segment_summary["Segment_ID"].isin(segment_ids)]
    max_pct = territory_df["Max_pct_AS_Clipped"].max() if not territory_df.empty else np.nan
    max_grade = territory_df["Stenosis_Severity_Grade"].max() if not territory_df.empty else pd.NA
    max_pct_for_flags = 0 if pd.isna(max_pct) else float(max_pct)

    territory_rows.append(
        {
            "Territory": territory_name,
            "Segments": ", ".join(str(x) for x in sorted(segment_ids)),
            "Highest_Stenosis_Location": top_segment_label(territory_df),
            "Territory_Max_pct_AS": max_pct,
            "Territory_Max_Grade": max_grade,
            "Obstructive_50plus": max_pct_for_flags >= 50,
            "Severe_70plus": max_pct_for_flags >= 70,
        }
    )

territory_summary = pd.DataFrame(territory_rows)
vessel_obstructive_count_50 = int(territory_summary["Obstructive_50plus"].sum())
vessel_severe_count_70 = int(territory_summary["Severe_70plus"].sum())
three_vessel_severe_70 = bool(territory_summary["Severe_70plus"].all())

lm_df = segment_summary.loc[segment_summary["Segment_ID"] == LEFT_MAIN_SEGMENT_ID]
lm_max_pct = lm_df["Max_pct_AS_Clipped"].max() if not lm_df.empty else np.nan
lm_stenosis_50plus = bool(pd.notna(lm_max_pct) and float(lm_max_pct) >= 50)

unmapped_for_territories = segment_summary.loc[
    ~segment_summary["Segment_ID"].isin(KNOWN_CADRADS_SEGMENTS)
].copy()

print(f"Obstructive vessel count (>=50%): {vessel_obstructive_count_50}")
print(f"Severe vessel count (>=70%)     : {vessel_severe_count_70}")
print(f"Left main >=50%                 : {lm_stenosis_50plus}")

if not unmapped_for_territories.empty:
    print(
        "Warning: these segments are included in global maximum/SIS calculations "
        "but are not assigned to RCA/LAD/LCX territory counts:"
    )
    display(unmapped_for_territories[["Segment_ID", "Segment_Name", "Max_pct_AS_Clipped"]])

display(territory_summary)

Obstructive vessel count (>=50%): 3
Severe vessel count (>=70%)     : 3
Left main >=50%                 : False


,Segment_ID,Segment_Name,Max_pct_AS_Clipped
4,19,Unmapped segment 19,83.381803


,Territory,Segments,Highest_Stenosis_Location,Territory_Max_pct_AS,Territory_Max_Grade,Obstructive_50plus,Severe_70plus
0,RCA,"1, 2, 3, 4",Right PDA (segment 4),87.248610,4,True,True
1,LAD,"6, 7, 8, 9, 10",Second diagonal (D2) (segment 10),92.800950,4,True,True
2,LCX,"11, 12, 13, 15, 17",LCX posterolateral branch (segment 17),93.278182,4,True,True


<a id="step4"></a>
# <p style="background-color:green; font-family:calibri; color:white; font-size:120%; text-align:center; border-radius:20px 20px;">4. Patient-Level CAD-RADS Logic</p>

Key JCCT 2022 statements used here:

- CAD-RADS is applied **on a per-patient basis** using the clinically most relevant, usually highest-grade, stenosis.
- CAD-RADS `0-3` correspond to maximal stenosis intervals `0%`, `1-24%`, `25-49%`, and `50-69%`.
- CAD-RADS `4A` corresponds to severe `70-99%` stenosis in one or two vessels.
- CAD-RADS `4B` corresponds to **left main >=50%** or **three-vessel obstructive disease >=70%**.
- CAD-RADS `5` corresponds to **100% total coronary occlusion**.

Precedence used in this implementation: total occlusion (`CAD-RADS 5`) is evaluated first because Table 4 defines it as a separate patient-level category. If no total occlusion is present, the 4B criteria are evaluated before the simple maximum-category rule so that left main `>=50%` is upgraded appropriately.

In [5]:
# Purpose: assign the final patient-level CAD-RADS numeric category/code.
assessable_segments = segment_summary.dropna(subset=["Max_pct_AS_Clipped"]).copy()
if assessable_segments.empty:
    raise ValueError("No assessable segment stenosis values were found.")

highest_row = assessable_segments.sort_values(
    ["Max_pct_AS_Clipped", "Segment_ID"], ascending=[False, True]
).iloc[0]
highest_stenosis_pct = float(highest_row["Max_pct_AS_Clipped"])
highest_stenosis_location = (
    f"{highest_row['Segment_Name']} (segment {int(highest_row['Segment_ID'])})"
)
max_segment_grade = int(assessable_segments["Stenosis_Severity_Grade"].max())
has_total_occlusion = bool((assessable_segments["Max_pct_AS_Clipped"] >= 100).any())

if has_total_occlusion:
    cad_rads_category = "5"
    cad_rads_rationale = "At least one segment has 100% stenosis: total coronary occlusion."
elif lm_stenosis_50plus:
    cad_rads_category = "4B"
    cad_rads_rationale = "Left main stenosis is >=50%, meeting the CAD-RADS 4B criterion."
elif three_vessel_severe_70:
    cad_rads_category = "4B"
    cad_rads_rationale = "RCA, LAD, and LCX each have at least one segment with >=70% stenosis."
elif max_segment_grade <= 3:
    cad_rads_category = str(max_segment_grade)
    cad_rads_rationale = f"Highest segment severity grade is {max_segment_grade}."
elif max_segment_grade == 4:
    cad_rads_category = "4A"
    cad_rads_rationale = "Severe stenosis is present, but CAD-RADS 4B criteria are not met."
else:
    cad_rads_category = str(max_segment_grade)
    cad_rads_rationale = f"Highest segment severity grade is {max_segment_grade}."

print(f"Highest segment grade      : {max_segment_grade}")
print(f"Highest stenosis location  : {highest_stenosis_location}")
print(f"Highest stenosis %AS       : {highest_stenosis_pct:.2f}%")
print(f"Patient CAD-RADS category  : {cad_rads_category}")
print(f"Rationale                  : {cad_rads_rationale}")

Highest segment grade      : 4
Highest stenosis location  : LCX posterolateral branch (segment 17)
Highest stenosis %AS       : 93.28%
Patient CAD-RADS category  : 4B
Rationale                  : RCA, LAD, and LCX each have at least one segment with >=70% stenosis.


<a id="step5"></a>
# <p style="background-color:green; font-family:calibri; color:white; font-size:120%; text-align:center; border-radius:20px 20px;">5. Plaque Burden Modifier (P)</p>

The JCCT paper describes plaque burden as an additional descriptor and lists SIS (Segment Involvement Score) categories in Table 2:

- `P1`: mild plaque burden, SIS `<=2`.
- `P2`: moderate plaque burden, SIS `3-4`.
- `P3`: severe plaque burden, SIS `5-7`.
- `P4`: extensive plaque burden, SIS `>=8`.

Note: Table 2 uses `<=2` and `>=8`. This avoids boundary gaps at SIS `2` and `8`. CAD-RADS 0 denotes absence of stenosis or plaque, so no `P0` modifier is required.

In [6]:
# Purpose: compute SIS and assign the CAD-RADS plaque burden modifier.
plaque_segments = segment_summary.loc[segment_summary["Max_pct_AS_Clipped"] > 0].copy()
sis_score = int(plaque_segments["Segment_ID"].nunique())

if sis_score == 0:
    plaque_modifier = None
    plaque_category = "No visible plaque/stenosis"
elif sis_score <= 2:
    plaque_modifier = "P1"
    plaque_category = "Mild plaque burden"
elif sis_score <= 4:
    plaque_modifier = "P2"
    plaque_category = "Moderate plaque burden"
elif sis_score <= 7:
    plaque_modifier = "P3"
    plaque_category = "Severe plaque burden"
else:
    plaque_modifier = "P4"
    plaque_category = "Extensive plaque burden"

final_cad_rads_code = f"CAD-RADS {cad_rads_category}"
if plaque_modifier is not None:
    final_cad_rads_code = f"{final_cad_rads_code}/{plaque_modifier}"

print(f"SIS score              : {sis_score} / {EXPECTED_SIS_SEGMENT_COUNT}")
print(f"Plaque modifier        : {plaque_modifier if plaque_modifier else 'Not required'}")
print(f"Plaque burden category : {plaque_category}")
print(f"Final CAD-RADS code    : {final_cad_rads_code}")

SIS score              : 14 / 16
Plaque modifier        : P4
Plaque burden category : Extensive plaque burden
Final CAD-RADS code    : CAD-RADS 4B/P4


<a id="step6"></a>
# <p style="background-color:green; font-family:calibri; color:white; font-size:120%; text-align:center; border-radius:20px 20px;">6. Priority Risk Score</p>

CAD-RADS itself is the clinical reporting category. For project triage and downstream automation, this notebook also creates a simple **priority risk score** derived directly from the final CAD-RADS category.

This score is not an official JCCT modifier. It is an internal project field to make patient-level outputs easier to sort.

In [7]:
# Purpose: assign an internal project priority score from the final CAD-RADS category.
RISK_PRIORITY_MAP = {
    "0": {"Priority_Risk_Score": 0, "Priority_Risk_Label": "Very low priority"},
    "1": {"Priority_Risk_Score": 1, "Priority_Risk_Label": "Low priority"},
    "2": {"Priority_Risk_Score": 2, "Priority_Risk_Label": "Low-moderate priority"},
    "3": {"Priority_Risk_Score": 3, "Priority_Risk_Label": "Moderate priority"},
    "4A": {"Priority_Risk_Score": 4, "Priority_Risk_Label": "High priority"},
    "4B": {"Priority_Risk_Score": 5, "Priority_Risk_Label": "Very high priority"},
    "5": {"Priority_Risk_Score": 6, "Priority_Risk_Label": "Critical priority"},
}

priority = RISK_PRIORITY_MAP[cad_rads_category]
priority_risk_score = priority["Priority_Risk_Score"]
priority_risk_label = priority["Priority_Risk_Label"]

print(f"Priority risk score : {priority_risk_score}")
print(f"Priority risk label : {priority_risk_label}")

Priority risk score : 5
Priority risk label : Very high priority


<a id="step7"></a>
# <p style="background-color:green; font-family:calibri; color:white; font-size:120%; text-align:center; border-radius:20px 20px;">7. Patient ID Card</p>

The card below is the final clinical summary for this phase. It is intentionally compact: the detailed evidence remains in the segment and territory tables.

In [8]:
# Purpose: generate a compact patient-level CAD-RADS report card.
patient_report = pd.DataFrame(
    [
        {
            "Sample_Name": SAMPLE_NAME,
            "Final_CAD_RADS_Code": final_cad_rads_code,
            "CAD_RADS_Category": cad_rads_category,
            "CAD_RADS_Rationale": cad_rads_rationale,
            "Highest_Stenosis_Location": highest_stenosis_location,
            "Highest_Stenosis_pct_AS": highest_stenosis_pct,
            "Vessel_Involvement_Count_50plus": vessel_obstructive_count_50,
            "Severe_Vessel_Count_70plus": vessel_severe_count_70,
            "Left_Main_50plus": lm_stenosis_50plus,
            "Three_Vessel_Severe_70plus": three_vessel_severe_70,
            "SIS_Score": sis_score,
            "SIS_Denominator": EXPECTED_SIS_SEGMENT_COUNT,
            "Plaque_Modifier": plaque_modifier if plaque_modifier else "Not required",
            "Plaque_Category": plaque_category,
            "Priority_Risk_Score": priority_risk_score,
            "Priority_Risk_Label": priority_risk_label,
        }
    ]
)

card_html = f"""
<div style="border:2px solid #2e7d32; border-radius:16px; padding:18px; max-width:760px; font-family:Calibri, Arial, sans-serif;">
  <h2 style="margin-top:0; color:#2e7d32;">Patient ID Card - {SAMPLE_NAME}</h2>
  <h1 style="margin:8px 0; color:#1b5e20;">{final_cad_rads_code}</h1>
  <p><b>Highest stenosis location:</b> {highest_stenosis_location} ({highest_stenosis_pct:.2f}% AS)</p>
  <p><b>Obstructive vessel count (>=50%):</b> {vessel_obstructive_count_50}</p>
  <p><b>Severe vessel count (>=70%):</b> {vessel_severe_count_70}</p>
  <p><b>SIS:</b> {sis_score} / {EXPECTED_SIS_SEGMENT_COUNT} - {plaque_modifier if plaque_modifier else 'No P modifier'} ({plaque_category})</p>
  <p><b>Priority risk:</b> {priority_risk_score} - {priority_risk_label}</p>
  <hr>
  <p><b>CAD-RADS rationale:</b> {cad_rads_rationale}</p>
</div>
"""

display(HTML(card_html))
display(patient_report)

,Sample_Name,Final_CAD_RADS_Code,CAD_RADS_Category,CAD_RADS_Rationale,Highest_Stenosis_Location,Highest_Stenosis_pct_AS,Vessel_Involvement_Count_50plus,Severe_Vessel_Count_70plus,Left_Main_50plus,Three_Vessel_Severe_70plus,SIS_Score,SIS_Denominator,Plaque_Modifier,Plaque_Category,Priority_Risk_Score,Priority_Risk_Label
0,Normal_1,CAD-RADS 4B/P4,4B,"RCA, LAD, and LCX each have at least one segme...",LCX posterolateral branch (segment 17),93.278182,3,3,False,True,14,16,P4,Extensive plaque burden,5,Very high priority


<a id="step8"></a>
# <p style="background-color:green; font-family:calibri; color:white; font-size:120%; text-align:center; border-radius:20px 20px;">8. Export Patient-Level Report</p>

The final patient-level report is exported to:

`results/block3_results/cad-rads/{sample_name}/cad_rads_report_{sample_name}.xlsx`

The workbook contains these sheets:

- `patient_report`: final CAD-RADS code, SIS, plaque modifier, and priority risk score.
- `territory_summary`: RCA/LAD/LCX involvement used for 4A/4B decisions.
- `segment_summary`: the Phase 3.2 segment table used as evidence.
- `unmapped_segments`: only written when the sample contains segment IDs outside the current RCA/LAD/LCX/LM grouping.

In [ ]:
# Purpose: export the patient-level CAD-RADS report and supporting evidence tables.
output_dir = OUTPUT_ROOT / SAMPLE_NAME
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / f"cad_rads_report_{SAMPLE_NAME}.xlsx"

with pd.ExcelWriter(output_path) as writer:
    patient_report.to_excel(writer, sheet_name="patient_report", index=False)
    territory_summary.to_excel(writer, sheet_name="territory_summary", index=False)
    segment_summary.to_excel(writer, sheet_name="segment_summary", index=False)
    if not unmapped_for_territories.empty:
        unmapped_for_territories.to_excel(writer, sheet_name="unmapped_segments", index=False)

print("Export complete:")
print(output_path)

Export complete:
C:\Users\adria\OneDrive\Escriptori\UPF'\TFG\UPF_TFGRepository\results\block3_results\cad-rads\Normal_2\cad_rads_report_Normal_2.xlsx
